In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 1. Importação das bibliotecas

In [ ]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    classification_report,
    confusion_matrix
)

## 2. Leitura do dataset

In [ ]:
caminhos = [
    "/kaggle/input/datasets/uciml/iris/Iris.csv",
    "Iris.csv"
]

caminho_iris = next((c for c in caminhos if os.path.exists(c)), None)

if caminho_iris is None:
    raise FileNotFoundError(
        "Iris.csv não foi encontrado. Coloque o arquivo na pasta do notebook."
    )

df = pd.read_csv(caminho_iris)

print("Arquivo utilizado:", caminho_iris)
print("Dimensão:", df.shape)

display(df.head())

In [ ]:
print("Colunas:")
print(df.columns.tolist())

print("\nQuantidade por espécie:")
print(df["Species"].value_counts())

print("\nValores ausentes:")
print(df.isnull().sum())

## 3. Separação X e y

In [ ]:
X = df.drop(columns=["Species", "Id"])
y = df["Species"]

print("X:")
display(X.head())

print("\ny:")
display(y.head())

## 4. Divisão treino/teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 5.  4 modelos sem GridSearchCV

In [ ]:
modelo_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

modelo_arvore = DecisionTreeClassifier(random_state=42)

modelo_floresta = RandomForestClassifier(random_state=42)

modelo_boosting = AdaBoostClassifier(random_state=42)

In [ ]:
def avaliar_modelo(nome, modelo):
    modelo.fit(X_train, y_train)

    previsoes = modelo.predict(X_test)

    acuracia = accuracy_score(y_test, previsoes)
    sensibilidade = recall_score(
        y_test,
        previsoes,
        average="macro"
    )

    print(f"===== {nome} =====")
    print(f"Acurácia: {acuracia:.4f} ({acuracia:.2%})")
    print(f"Sensibilidade (recall macro): {sensibilidade:.4f} ({sensibilidade:.2%})")

    print("\nRelatório de classificação:")
    print(classification_report(y_test, previsoes))

    print("Matriz de confusão:")
    print(confusion_matrix(y_test, previsoes))
    print()

    return acuracia, sensibilidade


resultados_sem_grid = {}

resultados_sem_grid["SVM"] = avaliar_modelo("SVM", modelo_svm)
resultados_sem_grid["Árvore de Decisão"] = avaliar_modelo(
    "Árvore de Decisão", modelo_arvore
)
resultados_sem_grid["Floresta Aleatória"] = avaliar_modelo(
    "Floresta Aleatória", modelo_floresta
)
resultados_sem_grid["Boosting (AdaBoost)"] = avaliar_modelo(
    "Boosting (AdaBoost)", modelo_boosting
)

## 6. GridSearchCV — SVM

In [ ]:
parametros_svm = [
    {
        "svc__C": [0.1, 1, 10, 100],
        "svc__kernel": ["linear"]
    },
    {
        "svc__C": [0.1, 1, 10, 100],
        "svc__kernel": ["rbf"],
        "svc__gamma": ["scale", "auto"]
    }
]

grid_svm = GridSearchCV(
    estimator=modelo_svm,
    param_grid=parametros_svm,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("Melhores parâmetros do SVM:")
print(grid_svm.best_params_)
print(f"Melhor acurácia média na validação cruzada: {grid_svm.best_score_:.4f}")

previsoes_svm_grid = grid_svm.predict(X_test)

acuracia_svm_grid = accuracy_score(y_test, previsoes_svm_grid)
sensibilidade_svm_grid = recall_score(
    y_test,
    previsoes_svm_grid,
    average="macro"
)

print(f"Acurácia no teste: {acuracia_svm_grid:.4f} ({acuracia_svm_grid:.2%})")
print(
    f"Sensibilidade no teste: "
    f"{sensibilidade_svm_grid:.4f} ({sensibilidade_svm_grid:.2%})"
)

## 7. GridSearchCV — Árvore de Decisão

In [ ]:
parametros_arvore = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [None, 2, 3, 4, 5, 10],
    "min_samples_split": [2, 5, 10]
}

grid_arvore = GridSearchCV(
    estimator=modelo_arvore,
    param_grid=parametros_arvore,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_arvore.fit(X_train, y_train)

print("Melhores parâmetros da Árvore de Decisão:")
print(grid_arvore.best_params_)
print(f"Melhor acurácia média na validação cruzada: {grid_arvore.best_score_:.4f}")

previsoes_arvore_grid = grid_arvore.predict(X_test)

acuracia_arvore_grid = accuracy_score(y_test, previsoes_arvore_grid)
sensibilidade_arvore_grid = recall_score(
    y_test,
    previsoes_arvore_grid,
    average="macro"
)

print(f"Acurácia no teste: {acuracia_arvore_grid:.4f} ({acuracia_arvore_grid:.2%})")
print(
    f"Sensibilidade no teste: "
    f"{sensibilidade_arvore_grid:.4f} ({sensibilidade_arvore_grid:.2%})"
)

## 8. GridSearchCV — Floresta Aleatória

In [ ]:
parametros_floresta = {
    "n_estimators": [50, 100, 200],
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 5]
}

grid_floresta = GridSearchCV(
    estimator=modelo_floresta,
    param_grid=parametros_floresta,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_floresta.fit(X_train, y_train)

print("Melhores parâmetros da Floresta Aleatória:")
print(grid_floresta.best_params_)
print(f"Melhor acurácia média na validação cruzada: {grid_floresta.best_score_:.4f}")

previsoes_floresta_grid = grid_floresta.predict(X_test)

acuracia_floresta_grid = accuracy_score(
    y_test,
    previsoes_floresta_grid
)

sensibilidade_floresta_grid = recall_score(
    y_test,
    previsoes_floresta_grid,
    average="macro"
)

print(
    f"Acurácia no teste: "
    f"{acuracia_floresta_grid:.4f} ({acuracia_floresta_grid:.2%})"
)
print(
    f"Sensibilidade no teste: "
    f"{sensibilidade_floresta_grid:.4f} ({sensibilidade_floresta_grid:.2%})"
)

## 9. GridSearchCV — Boosting (AdaBoost)

In [ ]:
parametros_boosting = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0]
}

grid_boosting = GridSearchCV(
    estimator=modelo_boosting,
    param_grid=parametros_boosting,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_boosting.fit(X_train, y_train)

print("Melhores parâmetros do AdaBoost:")
print(grid_boosting.best_params_)
print(f"Melhor acurácia média na validação cruzada: {grid_boosting.best_score_:.4f}")

previsoes_boosting_grid = grid_boosting.predict(X_test)

acuracia_boosting_grid = accuracy_score(
    y_test,
    previsoes_boosting_grid
)

sensibilidade_boosting_grid = recall_score(
    y_test,
    previsoes_boosting_grid,
    average="macro"
)

print(
    f"Acurácia no teste: "
    f"{acuracia_boosting_grid:.4f} ({acuracia_boosting_grid:.2%})"
)
print(
    f"Sensibilidade no teste: "
    f"{sensibilidade_boosting_grid:.4f} ({sensibilidade_boosting_grid:.2%})"
)

## 10. Tabela de comparação

In [ ]:
comparacao = []

for nome, valores in resultados_sem_grid.items():
    comparacao.append({
        "Modelo": nome,
        "GridSearchCV": "Não",
        "Acurácia": valores[0],
        "Sensibilidade": valores[1]
    })

resultados_com_grid = {
    "SVM": (acuracia_svm_grid, sensibilidade_svm_grid),
    "Árvore de Decisão": (acuracia_arvore_grid, sensibilidade_arvore_grid),
    "Floresta Aleatória": (
        acuracia_floresta_grid,
        sensibilidade_floresta_grid
    ),
    "Boosting (AdaBoost)": (
        acuracia_boosting_grid,
        sensibilidade_boosting_grid
    )
}

for nome, valores in resultados_com_grid.items():
    comparacao.append({
        "Modelo": nome,
        "GridSearchCV": "Sim",
        "Acurácia": valores[0],
        "Sensibilidade": valores[1]
    })

df_comparacao = pd.DataFrame(comparacao)

display(
    df_comparacao.style.format({
        "Acurácia": "{:.2%}",
        "Sensibilidade": "{:.2%}"
    })
)

In [ ]:
print("===== MELHORES RESULTADOS =====")

melhor_acuracia = df_comparacao.loc[
    df_comparacao["Acurácia"].idxmax()
]

melhor_sensibilidade = df_comparacao.loc[
    df_comparacao["Sensibilidade"].idxmax()
]

print(
    f"Maior acurácia: {melhor_acuracia['Modelo']} "
    f"({'com GridSearchCV' if melhor_acuracia['GridSearchCV'] == 'Sim' else 'sem GridSearchCV'}) "
    f"- {melhor_acuracia['Acurácia']:.2%}"
)

print(
    f"Maior sensibilidade: {melhor_sensibilidade['Modelo']} "
    f"({'com GridSearchCV' if melhor_sensibilidade['GridSearchCV'] == 'Sim' else 'sem GridSearchCV'}) "
    f"- {melhor_sensibilidade['Sensibilidade']:.2%}"
)

## 11. Análise dos resultados

Os quatro modelos foram executados no dataset Iris, primeiro sem otimização de hiperparâmetros e depois utilizando GridSearchCV.

A acurácia representa a proporção de previsões corretas. A sensibilidade (recall) mede a capacidade de identificar corretamente as amostras de cada classe. Como o problema possui três classes, foi utilizado average="macro", que calcula o recall de cada classe e depois faz a média.

Neste experimento, o GridSearchCV foi configurado com scoring="accuracy". Portanto, os melhores hiperparâmetros foram escolhidos com base na acurácia média da validação cruzada. Depois disso, os modelos otimizados foram avaliados separadamente no conjunto de teste usando acurácia e sensibilidade.

A tabela acima permite comparar diretamente o efeito da otimização para cada algoritmo. O resultado final deve ser interpretado a partir dos valores obtidos na execução deste notebook, pois pequenas mudanças na divisão dos dados ou nos hiperparâmetros testados podem alterar os resultados.

## 12. Conclusão

O experimento permitiu comparar SVM, Árvore de Decisão, Floresta Aleatória e
AdaBoost no dataset Iris.

Todos os modelos foram executados em duas condições: sem `GridSearchCV` e com
otimização de hiperparâmetros por validação cruzada. A comparação utilizou
acurácia e sensibilidade como métricas principais.

A análise final deve considerar simultaneamente as duas métricas e observar
também a diferença entre os resultados obtidos antes e depois da otimização.
Dessa forma, é possível verificar na prática como a escolha dos hiperparâmetros
pode alterar o desempenho dos classificadores.